<a href="https://colab.research.google.com/github/dr-bankert-augustana/PHYS_200/blob/main/Pal_2_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a name="Notebook-Start"></a>

---

<font size = 7> <b> PAL (Linear Models Part 2) </b> </font>

---

In this PAL, we will be using the 'tips' dataset from the scikit-learn library.

<a name="Define-Useful-Functions"></a>

---

<font size = 6> <b> Define Useful Functions </b> </font>

---

In [ ]:
#@title This cell defines the functions: load_model, display_dataframes, display_model, plot_data, rmse_loss.

##=============================================================================================##
## Included Functions:                                                                         ##
##                                                                                             ##
## 1. load_model         - Load and clean data from input file, split into feature and target  ##
## 2. display_dataframes - Display multiple DataFrames side-by-side with titles                ##
## 3. display_model      - Display models' parameters and loss in a DataFrame                  ##
## 4. plot_data          - Create a graph with raw data, can add model to graph if needed      ##
## 5. rmse_loss          - Implement the root mean squared error loss function                 ##
##=============================================================================================##

##=============================================================================================##
## Function:  load_model                                                                       ##
##                                                                                             ##
## Purpose:   Load and clean data from input file, split into feature and target               ##
##                                                                                             ##
## Input(s):  filename     - Name of the file containing the data                              ##
##            feature_list - List of column names containing feature data                      ##
##            target_list  - List of column names containing target data                       ##
##                                                                                             ##
## Output(s): features     - DataFrame containing model feature data                           ##
##            targets      - DataFrame containint model target data                            ##
##=============================================================================================##

def load_model(filename, feature_list, target_list):

  ##===========================================================================================##
  ## Load in the Full Data Set and Drop Any Rows Missing Data:                                 ##
  ##===========================================================================================##

  data = pd.read_csv(filename).dropna()

  ##===========================================================================================##
  ## Separate the Feature and Target Data and Drop Any Rows Missing Data:                      ##
  ##===========================================================================================##

  # Identify the feature data:

  features = pd.DataFrame(data[feature_list])

  # Identify the target data:

  targets = pd.DataFrame(data[target_list])

  # Display the feature and target data:

  display_dataframes([data, features, targets], ["Full Dataset", "Feature Data", "Target Data"])

  ##===========================================================================================##
  ## Return the Feature Data, and Target Data:                                                 ##
  ##===========================================================================================##

  return features, targets

##=============================================================================================##
## Function:  display_dataframes                                                               ##
##                                                                                             ##
## Purpose:   Display multiple DataFrames side-by-side with titles                             ##
##                                                                                             ##
## Input(s):  item_list  - List of DataFrames to be displayed                                  ##
##            title_list - List of titles for the displayed DataFrames                         ##
##                                                                                             ##
## Output(s): None                                                                             ##
##=============================================================================================##

def display_dataframes(item_list, title_list):

  ##===========================================================================================##
  ## Create a String for Housing the Commands to Be Sent to the display_html() Function:       ##
  ##===========================================================================================##

  html_str = ''

  ##===========================================================================================##
  ## Loop Over the Elements in the item_list and title_list:                                   ##
  ##===========================================================================================##

  for df, title in zip(item_list, title_list):

    ##=========================================================================================##
    ## Wrap title and DataFrames in a Styled HTML <div>:                                       ##
    ##=========================================================================================##

      html_str += f'''
      <div style="display: inline-block; margin-right: 20px; vertical-align: top;">
          <h3 style="text-align: center;">{title}</h3>
          {pd.DataFrame(df).head().to_html()}
      </div>
      '''

  ##===========================================================================================##
  ## Send the HTML String to the display_html() Function:                                      ##
  ##===========================================================================================##

  display_html(html_str, raw = True)

##=============================================================================================##
## Function:  display_model                                                                    ##
##                                                                                             ##
## Purpose:   Display models' parameters and loss in a DataFrame                               ##
##                                                                                             ##
## Input(s):  model_list - List of models' names to be displayed                               ##
##            coef_list  - List of models' coefficients to be displayed                        ##
##            bias_list  - List of models' bias to be displayed                                ##
##            loss_list  - List of models' loss to be displayed                                ##
##            title      - Title for the display of models                                     ##
##            truncation - Number of decimals to display for numbers (optional)                ##
##                                                                                             ##
## Output(s): None                                                                             ##
##=============================================================================================##

def display_model(model_list, coefficient_list, bias_list, loss_list, title, truncation = 3):

  ##===========================================================================================##
  ## Create a DataFrame to Hold the Results:                                                   ##
  ##===========================================================================================##

  results = pd.DataFrame()

  ##===========================================================================================##
  ## Round the Numeric Values to the Desired Level of Desired Truncation:                      ##
  ##===========================================================================================##

  # Round the coefficients:

  for i in range(len(coefficient_list)):

    coefficient_list[i] = np.round(coefficient_list[i], truncation)

  # Round the biases:

  for i in range(len(bias_list)):

    bias_list[i] = np.round(bias_list[i], truncation)

  # Round the losses:

  for i in range(len(loss_list)):

    loss_list[i] = np.round(loss_list[i], truncation)

  ##===========================================================================================##
  ## Add the Contents of the DataFrame Columns:                                                ##
  ##===========================================================================================##

  # Add the model names:

  results["Model"] = model_list

  # Add the model coefficients:

  results["Coefficient(s)"] = coefficient_list

  # Add the model biases:

  results["Bias(es)"] = bias_list

  # Add the model rmses:

  results["Loss"] = loss_list

  ##===========================================================================================##
  ## Index the Results DataFrame By Model Name:                                                ##
  ##===========================================================================================##

  results.set_index("Model", inplace = True)

  ##===========================================================================================##
  ## Display the Results DataFrame Using display_dataframes():                                 ##
  ##===========================================================================================##

  display_dataframes([results], [title])

##=============================================================================================##
## Function:  plot_data                                                                        ##
##                                                                                             ##
## Purpose:   Create a scatterplot with optional model overlays and error bands                ##
##                                                                                             ##
## Input(s):  x_data        - List of data points' x-axis values                               ##
##            y_data        - List of data points' y-axis values                               ##
##            title         - Graph title                                                      ##
##            axis_labels   - Override axis labels [x_label, y_label] (optional)               ##
##            model_list    - List of model predictions to overlay (optional)                  ##
##            color_list    - Colors for each model line (optional)                            ##
##            label_list    - Labels for each model line (optional)                            ##
##            error_display - Show +/- error band around first model (default is False)        ##
##            error         - Error value for shaded band (optional)                           ##
##                                                                                             ##
## Output(s): graph       - Matplotlib axes object, can be used for overplotting               ##
##=============================================================================================##

def plot_data(x_data, y_data, title, axis_labels = [], model_list = [], color_list = [],
              label_list = [], error_display = False, error = 0):

  ##===========================================================================================##
  ## Setup the Graph:                                                                          ##
  ##===========================================================================================##

  # Create the Matplotlib figure:

  figure = plt.figure(figsize = (12, 9))

  # Add a graph to the figure:

  graph = figure.add_subplot()

  # Set the graph background Color:

  graph.set_facecolor('lightcyan')

  # Set the graph title:

  graph.set_title(title, fontsize = 20)

  # Set the x_label and y_label:

  if (axis_labels != []):

    graph.set_xlabel(axis_labels[0], fontsize = 14)

    graph.set_ylabel(axis_labels[1], fontsize = 14)

  # Apply a grid to the graph:

  graph.grid(which = 'both')

  # Adjust the x-axis scale of the graph:

  graph.autoscale(enable = True, axis = 'x', tight = False)

  # Adjust the y-axis scale of the graph:

  graph.autoscale(enable = True, axis = 'y', tight = False)

  ##===========================================================================================##
  ## Add Data to the Graph:                                                                    ##
  ##===========================================================================================##

  # Create a scatterplot of the data:

  sns.scatterplot(x = x_data, y = y_data, ax = graph)

  # Overlay model predictions:

  for i in range(0, len(model_list)):

    graph.plot(x_data, model_list[i], color = color_list[i], label = label_list[i])

  ##===========================================================================================##
  ## If requested, show the +/- error bounds:                                                  ##
  ##===========================================================================================##

  if ((error_display == True) and model_list != []):

    # Create the error+ model:

    model_plus_error  = model_list[0].iloc[:, 0] + error

    # Create the error- model:

    model_minus_error = model_list[0].iloc[:, 0] - error

    # Store the error+ and error- models in a DataFrame and Sort by x_data values:

    error_df = pd.DataFrame({
        'X-Data': x_data,
        'Model+Error': model_plus_error,
        'Model-Error': model_minus_error
    }).sort_values(by = "X-Data")

    # Use graph.fill to highlight the region between the error+ and error- models:

    graph.fill_between(error_df['X-Data'], error_df['Model+Error'], error_df['Model-Error'],
                       alpha = 0.5, color = (0.6, 0.6, 0.6), label = "Error Bounds")

  ##===========================================================================================##
  ## Apply the Legend and Return the graph Object:                                             ##
  ##===========================================================================================##

  # Add the graph legend:

  if (label_list != []): graph.legend()

  # Return the graph:

  return graph

##=============================================================================================##
## Function:  rmse_loss                                                                        ##
##                                                                                             ##
## Purpose:   Implement the root mean squared error loss function                              ##
##                                                                                             ##
## Input(s):  X           - feature (independent) data                                         ##
##            y           - target (dependent) data                                            ##
##            coefficient - slope of the linear model                                          ##
##            bias        - bias of the linear model                                           ##
##                                                                                             ##
## Output(s): loss        - root mean squared error loss                                       ##
##=============================================================================================##

def rmse_loss(X, y, coefficient, bias):

  ##===========================================================================================##
  ## Create the Model's Predictions Based on a Linear Combination of the Feature Data:         ##
  ##===========================================================================================##

  predictions = X * coefficient + bias

  ##===========================================================================================##
  ## Calculate the Root Mean Squared Error Loss:                                               ##
  ##===========================================================================================##

  # Calculate the squared error:

  squared_error = (y - predictions)**2

  # Find the mean of the squared error:

  mse = squared_error.mean()

  # Take the square root of the mean squared error:

  loss = np.sqrt(mse)

  # Return the root mean squared error loss:

  return loss

<a name="Data"></a>

---

<font size = 6> <b> The Data </b> </font>

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display_html

tips_df = sns.load_dataset("tips")

display(tips_df.head())

<a name="Univariate"></a>

---

<font size = 6> <b> Part 1: Univariate Linear Model (total bill) </b> </font>

---

In this part, we will construct a linear model that predicts the tip left by a table based on the total bill of the meal.

<font size = 5> <b> Step 1: Identify the Target and Feature Data </b> </font>

In [ ]:
# Set the feature list:

feature = # ["feature_column_name"]

# Set the target list:

target = # ["target_column_name"]

# Isolate the Feature and Target as individual DataFrames:

X = pd.DataFrame(tips_df[feature])

y = pd.DataFrame(tips_df[target])

display(X)
print()
display(y)

<font size = 5> <b> Step 2: Create With-bias and No-bias models using LinearRegression </b> </font>

In [ ]:
##=============================================================================================##
## Use a LinearRegression Object to Find the Best Fit for the Model:                           ##
##=============================================================================================##

# Create a LinearRegression object without a forced-origin intercept:

model_1 =

# Create a LinearRegression object with a forced-origin intercept:

model_2 =

# Fit the LinearRegression objects to the features and target:

model_1.fit(X, y)

model_2.fit(X, y)

# Get the linear coefficient and bias for model 1:

model_1_coef =
model_1_bias =

# Get the linear coefficient and bias for model 2:

model_2_coef =
model_2_bias =

##=============================================================================================##
## Get The Models' Predictions:                                                                ##
##=============================================================================================##

model_1_predictions =

model_2_predictions =

##=============================================================================================##
## Calculate Models' RMSE Loss:                                                                ##
##=============================================================================================##

model_1_loss =

model_2_loss =

##=============================================================================================##
## Display The Results:                                                                        ##
##=============================================================================================##

# Set the model names:

model_list = ["Model With Bias", "Model Without Bias"]

# Set the coefficient(s) list:

coef_list = [model_1_coef, model_2_coef]

# Set the bias list:

bias_list = [model_1_bias, model_2_bias]

# Set the loss list:

loss_list = [model_1_loss, model_2_loss]

# Display the results:

display_model(model_list, coef_list, bias_list, loss_list, "Comparing Models")

<font size = 5> <b> Step 3: Interpret the Results </b> </font>

1. What is the physical meaning behind each model?

2. Which model has the lowest Loss?

3. Which model makes the most physical sense?

4. Which is the best model to use and why?

<a name="Univariate"></a>

---

<font size = 6> <b> Part 2: Univariate Linear Model (table size) </b> </font>

---

In this part, we will construct a linear model that predicts the tip left by a table based on the number of people at the table.

<font size = 5> <b> Step 1: Identify the Target and Feature Data </b> </font>

<font size = 5> <b> Step 2: Create With-bias and No-bias models using LinearRegression </b> </font>

<font size = 5> <b> Step 3: Interpret the Results </b> </font>

1. What is the physical meaning behind each model?

2. Which model has the lowest Loss?

3. Which model makes the most physical sense?

4. Which is the best model to use and why?

<a name="Univariate"></a>

---

<font size = 6> <b> Part 3: Multivariate Linear Model </b> </font>

---

In this part, we will construct a linear model that predicts the tip left by a table based on the total bill of the meal and the number of people at the table.

<font size = 5> <b> Step 1: Identify the Target and Feature Data </b> </font>

<font size = 5> <b> Step 2: Create With-bias and No-bias models using LinearRegression </b> </font>

<font size = 5> <b> Step 3: Interpret the Results </b> </font>

1. What is the physical meaning behind each model?

2. Which model has the lowest Loss?

3. Which model makes the most physical sense?

4. Which is the best model to use and why?

5. Compare the univariate and multivariate models, what does this tell us about the tipping habits of people in restaurants?